In [1]:
import logging
import sys

logger = logging.getLogger("logger")
logger.setLevel(logging.INFO)

# Avoid duplicate handlers if you re-run the cell
if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(name)s | %(message)s",
        datefmt="%H:%M:%S",
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.info("Logger is ready")


14:00:47 | INFO | logger | Logger is ready


In [2]:
from pathlib import Path
import pandas as pd
from PyDI.io import load_parquet
import re

ROOT = Path.cwd().parent

DATA_DIR = ROOT / "parquet"
OUTPUT_DIR = ROOT / "output"
MLDS_DIR = ROOT / "ml-datasets"
BLOCK_EVAL_DIR = OUTPUT_DIR / "blocking_evaluation"
CORR_DIR = OUTPUT_DIR / "correspondences"

PIPELINE_DIR = OUTPUT_DIR / "data_fusion"
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

amazon_sample = load_parquet(
    DATA_DIR / "amazon_sample.parquet",
    name="amazon_sample"
)

goodreads_sample = load_parquet(
    DATA_DIR / "goodreads_sample.parquet",
    name="goodreads_sample"
)

metabooks_sample = load_parquet(
  DATA_DIR / "metabooks_sample.parquet",
  name="metabooks_sample"
)

def clean_text(t):
    t = str(t).lower()
    t = re.sub(r'<.*?>', '', t)
    # only change: space out dots between letters so initials stay separate
    t = re.sub(r'(?<=\b[a-z])\.\s*(?=[a-z]\b)', ' ', t)
    t = re.sub(r'[^a-z0-9\s]', '', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t


def clean_author_field_goodreads(author_str: str) -> str:
    if not isinstance(author_str, str):
        return ""
    authors = []
    parts = [p.strip() for p in author_str.split(",") if p.strip()]
    for i, raw in enumerate(parts):
        has_paren = "(" in raw
        name = re.split(r"\s*\(", raw)[0].strip()  # drop parenthetical
        if len(name.split()) < 2:  # need first + surname
            continue
        if i == 0:
            authors.append(name)
            continue
        if has_paren:
            break  # stop at first later author with parentheses
        authors.append(name)
    return ", ".join(authors)


amazon_sample['clean_title'] = amazon_sample['title'].apply(clean_text)
goodreads_sample['clean_title'] = goodreads_sample['title'].apply(clean_text)
metabooks_sample['clean_title'] = metabooks_sample['title'].apply(clean_text)
amazon_sample["clean_author"] = amazon_sample["author"].apply(clean_text)
# We have an extra step for goodreads authors since it contains unnecessary author info in the author field
goodreads_sample["clean_author"] = goodreads_sample["author"].apply(clean_author_field_goodreads)
goodreads_sample["clean_author"] = goodreads_sample["clean_author"].apply(clean_text)
metabooks_sample["clean_author"] = metabooks_sample["author"].apply(clean_text)
amazon_sample["clean_publisher"] = amazon_sample["publisher"].apply(clean_text)
metabooks_sample["clean_publisher"] = metabooks_sample["publisher"].apply(clean_text)
goodreads_sample["clean_publisher"] = goodreads_sample["publisher"].apply(clean_text)

In [3]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Tuple
from collections import Counter
import re

import numpy as np
import pandas as pd

from gensim.corpora import Dictionary
from gensim.models import TfidfModel
from gensim.similarities import SparseMatrixSimilarity


# -----------------------------
# Prep
# -----------------------------
def _safe_str(x) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip()


def _prep_two_df(df: pd.DataFrame, side: str) -> pd.DataFrame:
    out = df.copy()
    out["__side"] = side
    for c in ["clean_title", "clean_author", "clean_publisher"]:
        if c not in out.columns:
            out[c] = ""
        out[c] = out[c].map(_safe_str)
    if "publish_year" not in out.columns:
        out["publish_year"] = np.nan
    if "isbn_clean" not in out.columns:
        out["isbn_clean"] = ""
    out["isbn_clean"] = out["isbn_clean"].map(_safe_str)
    return out


# -----------------------------
# Tokenisation (NO deletions)
# - does not drop rare words
# - does not drop numbers
# - does not drop punctuation tokens explicitly; it simply extracts "word-like" tokens
# -----------------------------
_WORD_RE = re.compile(r"[0-9A-Za-z]+(?:'[0-9A-Za-z]+)?", re.UNICODE)


def _tokenize_keep_all_words(text: str) -> List[str]:
    if text is None:
        return []
    s = str(text).lower().strip()
    if not s:
        return []
    return _WORD_RE.findall(s)


# -----------------------------
# Gensim TF-IDF index per field (NO filter_extremes)
# -----------------------------
@dataclass
class GensimFieldIndex:
    dictionary: Dictionary
    tfidf: TfidfModel
    index: SparseMatrixSimilarity
    num_right: int


def _build_gensim_tfidf_index_no_deletions(
    right_texts: List[str],
    *,
    smartirs: str = "ntc",
) -> GensimFieldIndex:
    right_tokens = [_tokenize_keep_all_words(t) for t in right_texts]
    dictionary = Dictionary(right_tokens)
    dictionary.compactify()  # reindex only; does NOT delete words

    num_right = len(right_tokens)

    if len(dictionary) == 0:
        dummy_tfidf = TfidfModel([[]], dictionary=dictionary, smartirs=smartirs)
        dummy_index = SparseMatrixSimilarity([[]], num_features=1)
        return GensimFieldIndex(dictionary=dictionary, tfidf=dummy_tfidf, index=dummy_index, num_right=num_right)

    right_bow = [dictionary.doc2bow(toks) for toks in right_tokens]
    tfidf = TfidfModel(right_bow, dictionary=dictionary, smartirs=smartirs)
    right_tfidf = tfidf[right_bow]
    index = SparseMatrixSimilarity(right_tfidf, num_features=len(dictionary))
    return GensimFieldIndex(dictionary=dictionary, tfidf=tfidf, index=index, num_right=num_right)


def _gensim_sims_for_left_text_no_deletions(field_index: GensimFieldIndex, left_text: str) -> np.ndarray:
    if field_index.num_right == 0:
        return np.zeros((0,), dtype=np.float32)
    if len(field_index.dictionary) == 0:
        return np.zeros((field_index.num_right,), dtype=np.float32)

    bow = field_index.dictionary.doc2bow(_tokenize_keep_all_words(left_text))
    vec = field_index.tfidf[bow]
    sims = field_index.index[vec]
    return np.asarray(sims, dtype=np.float32)


def _topk_from_sims(sims: np.ndarray, k: int) -> np.ndarray:
    n = sims.shape[0]
    if n == 0:
        return np.array([], dtype=np.int64)
    k = min(k, n)
    if k == n:
        return np.argsort(-sims)
    cand = np.argpartition(-sims, k)[:k]
    return cand[np.argsort(-sims[cand])]


def _year_equal(a, b) -> bool:
    if pd.isna(a) or pd.isna(b):
        return False
    try:
        return int(a) == int(b)
    except Exception:
        return str(a) == str(b)


# -----------------------------
# Thresholds + generator
# -----------------------------
@dataclass
class Thresholds:
    s1_title_min: float = 0.85   # scenario 1: title only
    s2_title_min: float = 0.80   # scenario 2: title + author
    s2_author_min: float = 0.75  # scenario 2: title + author
    s3_author_min: float = 0.85  # scenario 3: author + same year
    s4_title_min: float = 0.80   # scenario 4: title + same year


def generate_corner_case_pairs_gensim(
    left_df: pd.DataFrame,
    right_df: pd.DataFrame,
    *,
    scenario_counts: Dict[str, int],
    left_id_col: str = "id",
    right_id_col: str = "id",
    n_neighbors: int = 80,
    random_state: int = 42,
    smartirs: str = "ntc",
    weights: Tuple[float, float, float, float] = (0.45, 0.25, 0.15, 0.15),  # title, author, year, publisher
    thresholds: Thresholds = Thresholds(),
    max_attempts_multiplier: int = 30,
) -> pd.DataFrame:
    """
    Corner-case generator with gensim TF-IDF word unigrams, NO SVD, NO token deletions.

    Scenario names in scenario_counts:
      1_extremely_similar_titles
      2_similar_titles_and_similar_author
      3_extremely_similar_author_same_year
      4_similar_title_same_year
      5_weighted_all_fields
    """
    rng = np.random.default_rng(random_state)

    L = _prep_two_df(left_df, "left")
    R = _prep_two_df(right_df, "right")

    if left_id_col not in L.columns:
        L[left_id_col] = np.arange(len(L))
    if right_id_col not in R.columns:
        R[right_id_col] = np.arange(len(R))

    # Build RIGHT indices per field (no deletions)
    idx_title = _build_gensim_tfidf_index_no_deletions(R["clean_title"].tolist(), smartirs=smartirs)
    idx_author = _build_gensim_tfidf_index_no_deletions(R["clean_author"].tolist(), smartirs=smartirs)
    idx_publisher = _build_gensim_tfidf_index_no_deletions(R["clean_publisher"].tolist(), smartirs=smartirs)

    # Per-left caches (avoid recomputing full sim vectors)
    title_sims_cache: Dict[int, np.ndarray] = {}
    author_sims_cache: Dict[int, np.ndarray] = {}
    publisher_sims_cache: Dict[int, np.ndarray] = {}

    def get_title_sims(i: int) -> np.ndarray:
        if i not in title_sims_cache:
            title_sims_cache[i] = _gensim_sims_for_left_text_no_deletions(idx_title, L.loc[i, "clean_title"])
        return title_sims_cache[i]

    def get_author_sims(i: int) -> np.ndarray:
        if i not in author_sims_cache:
            author_sims_cache[i] = _gensim_sims_for_left_text_no_deletions(idx_author, L.loc[i, "clean_author"])
        return author_sims_cache[i]

    def get_publisher_sims(i: int) -> np.ndarray:
        if i not in publisher_sims_cache:
            publisher_sims_cache[i] = _gensim_sims_for_left_text_no_deletions(idx_publisher, L.loc[i, "clean_publisher"])
        return publisher_sims_cache[i]

    def year_equal(i: int, j: int) -> bool:
        return _year_equal(L.loc[i, "publish_year"], R.loc[j, "publish_year"])

    def metrics(i: int, j: int) -> Dict[str, float]:
        st = float(get_title_sims(i)[j]) if idx_title.num_right > 0 else 0.0
        sa = float(get_author_sims(i)[j]) if idx_author.num_right > 0 else 0.0
        sp = float(get_publisher_sims(i)[j]) if idx_publisher.num_right > 0 else 0.0
        ye = 1.0 if year_equal(i, j) else 0.0
        w0, w1, w2, w3 = weights
        sw = w0 * st + w1 * sa + w2 * ye + w3 * sp
        return {
            "sim_title": st,
            "sim_author": sa,
            "sim_publisher": sp,
            "year_equal": float(ye),
            "sim_weighted": float(sw),
        }

    def quota(name: str) -> int:
        return int(scenario_counts.get(name, 0))

    counter = Counter()
    used = set()
    rows = []

    def try_add(i: int, j: int, scenario: str, m: Dict[str, float]) -> bool:
        if quota(scenario) <= 0:
            return False
        if counter[scenario] >= quota(scenario):
            return False
        key = (int(i), int(j), scenario)
        if key in used:
            return False

        rows.append({
            "scenario": scenario,
            "id1": L.loc[i, left_id_col],
            "id2": R.loc[j, right_id_col],
            "left_isbn_clean": L.loc[i, "isbn_clean"],
            "right_isbn_clean": R.loc[j, "isbn_clean"],
            "left_title": L.loc[i, "clean_title"],
            "right_title": R.loc[j, "clean_title"],
            "left_author": L.loc[i, "clean_author"],
            "right_author": R.loc[j, "clean_author"],
            "left_publisher": L.loc[i, "clean_publisher"],
            "right_publisher": R.loc[j, "clean_publisher"],
            "left_year": L.loc[i, "publish_year"],
            "right_year": R.loc[j, "publish_year"],
            **m
        })
        used.add(key)
        counter[scenario] += 1
        return True

    def done(s: str) -> bool:
        return counter[s] >= quota(s)

    # Scenario 1
    s1 = "1_extremely_similar_titles"
    if quota(s1) > 0:
        attempts = 0
        max_attempts = quota(s1) * max_attempts_multiplier
        while not done(s1) and attempts < max_attempts:
            attempts += 1
            i = int(rng.integers(0, len(L)))
            sims = get_title_sims(i)
            topk = _topk_from_sims(sims, n_neighbors)
            for j in topk:
                if float(sims[j]) < thresholds.s1_title_min:
                    break
                if try_add(i, int(j), s1, metrics(i, int(j))):
                    break

    # Scenario 2
    s2 = "2_similar_titles_and_similar_author"
    if quota(s2) > 0:
        attempts = 0
        max_attempts = quota(s2) * max_attempts_multiplier
        while not done(s2) and attempts < max_attempts:
            attempts += 1
            i = int(rng.integers(0, len(L)))
            t_sims = get_title_sims(i)
            a_sims = get_author_sims(i)
            topk = _topk_from_sims(t_sims, n_neighbors)
            for j in topk:
                if float(t_sims[j]) < thresholds.s2_title_min:
                    break
                if float(a_sims[j]) < thresholds.s2_author_min:
                    continue
                if try_add(i, int(j), s2, metrics(i, int(j))):
                    break

    # Scenario 3
    s3 = "3_extremely_similar_author_same_year"
    if quota(s3) > 0:
        attempts = 0
        max_attempts = quota(s3) * max_attempts_multiplier
        while not done(s3) and attempts < max_attempts:
            attempts += 1
            i = int(rng.integers(0, len(L)))
            a_sims = get_author_sims(i)
            topk = _topk_from_sims(a_sims, n_neighbors)
            for j in topk:
                if float(a_sims[j]) < thresholds.s3_author_min:
                    break
                if not year_equal(i, int(j)):
                    continue
                if try_add(i, int(j), s3, metrics(i, int(j))):
                    break

    # Scenario 4
    s4 = "4_similar_title_same_year"
    if quota(s4) > 0:
        attempts = 0
        max_attempts = quota(s4) * max_attempts_multiplier
        while not done(s4) and attempts < max_attempts:
            attempts += 1
            i = int(rng.integers(0, len(L)))
            t_sims = get_title_sims(i)
            topk = _topk_from_sims(t_sims, n_neighbors)
            for j in topk:
                if float(t_sims[j]) < thresholds.s4_title_min:
                    break
                if not year_equal(i, int(j)):
                    continue
                if try_add(i, int(j), s4, metrics(i, int(j))):
                    break

    # Scenario 5
    s5 = "5_weighted_all_fields"
    if quota(s5) > 0:
        buckets = [(0.85, 0.92), (0.70, 0.85), (0.55, 0.70)]
        target = quota(s5)
        per_bucket = max(1, target // len(buckets))

        attempts = 0
        max_attempts = target * max_attempts_multiplier * 2

        for lo, hi in buckets:
            filled = 0
            while filled < per_bucket and not done(s5) and attempts < max_attempts:
                attempts += 1
                i = int(rng.integers(0, len(L)))
                t_sims = get_title_sims(i)
                topk = _topk_from_sims(t_sims, n_neighbors)
                for j in topk:
                    m = metrics(i, int(j))
                    if lo <= m["sim_weighted"] < hi:
                        if try_add(i, int(j), s5, m):
                            filled += 1
                            break

        while not done(s5) and attempts < max_attempts:
            attempts += 1
            i = int(rng.integers(0, len(L)))
            t_sims = get_title_sims(i)
            topk = _topk_from_sims(t_sims, n_neighbors)
            for j in topk:
                m = metrics(i, int(j))
                if m["sim_weighted"] >= 0.55:
                    if try_add(i, int(j), s5, m):
                        break

    return pd.DataFrame(rows)

In [5]:
counts = {
    "1_extremely_similar_titles": 100,
    "2_similar_titles_and_similar_author": 100,
    "3_extremely_similar_author_same_year": 50,
    "4_similar_title_same_year": 50,
    "5_weighted_all_fields": 200,
}

corner_case_m2a = generate_corner_case_pairs_gensim(
    metabooks_sample, amazon_sample,
    scenario_counts=counts,
    left_id_col="id",
    right_id_col="id",
    n_neighbors=120,
)
corner_case_m2g = generate_corner_case_pairs_gensim(
    metabooks_sample, goodreads_sample,
    scenario_counts=counts,
    left_id_col="id",
    right_id_col="id",
    n_neighbors=120,

)
corner_case_g2a = generate_corner_case_pairs_gensim(
    goodreads_sample, amazon_sample,
    scenario_counts=counts,
    left_id_col="id",
    right_id_col="id",
    n_neighbors=120,
)

In [7]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Tuple
from collections import Counter
import re

import numpy as np
import pandas as pd

from gensim.corpora import Dictionary
from gensim.models import TfidfModel
from gensim.similarities import SparseMatrixSimilarity


# -----------------------------
# Prep
# -----------------------------
def _safe_str(x) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip()


def _prep_two_df(df: pd.DataFrame, side: str) -> pd.DataFrame:
    out = df.copy()
    out["__side"] = side
    for c in ["clean_title", "clean_author", "clean_publisher"]:
        if c not in out.columns:
            out[c] = ""
        out[c] = out[c].map(_safe_str)
    if "publish_year" not in out.columns:
        out["publish_year"] = np.nan
    if "isbn_clean" not in out.columns:
        out["isbn_clean"] = ""
    out["isbn_clean"] = out["isbn_clean"].map(_safe_str)
    return out


# -----------------------------
# Tokenisation (NO deletions)
# -----------------------------
_WORD_RE = re.compile(r"[0-9A-Za-z]+(?:'[0-9A-Za-z]+)?", re.UNICODE)


def _tokenize_keep_all_words(text: str) -> List[str]:
    if text is None:
        return []
    s = str(text).lower().strip()
    if not s:
        return []
    return _WORD_RE.findall(s)


# -----------------------------
# Gensim TF-IDF index per field (NO filter_extremes)
# -----------------------------
@dataclass
class GensimFieldIndex:
    dictionary: Dictionary
    tfidf: TfidfModel
    index: SparseMatrixSimilarity
    num_right: int


def _build_gensim_tfidf_index_no_deletions(
    right_texts: List[str],
    *,
    smartirs: str = "ntc",
) -> GensimFieldIndex:
    right_tokens = [_tokenize_keep_all_words(t) for t in right_texts]
    dictionary = Dictionary(right_tokens)
    dictionary.compactify()

    num_right = len(right_tokens)

    if len(dictionary) == 0:
        dummy_tfidf = TfidfModel([[]], dictionary=dictionary, smartirs=smartirs)
        dummy_index = SparseMatrixSimilarity([[]], num_features=1)
        return GensimFieldIndex(dictionary=dictionary, tfidf=dummy_tfidf, index=dummy_index, num_right=num_right)

    right_bow = [dictionary.doc2bow(toks) for toks in right_tokens]
    tfidf = TfidfModel(right_bow, dictionary=dictionary, smartirs=smartirs)
    right_tfidf = tfidf[right_bow]
    index = SparseMatrixSimilarity(right_tfidf, num_features=len(dictionary))
    return GensimFieldIndex(dictionary=dictionary, tfidf=tfidf, index=index, num_right=num_right)


def _gensim_sims_for_left_text_no_deletions(field_index: GensimFieldIndex, left_text: str) -> np.ndarray:
    if field_index.num_right == 0:
        return np.zeros((0,), dtype=np.float32)
    if len(field_index.dictionary) == 0:
        return np.zeros((field_index.num_right,), dtype=np.float32)

    bow = field_index.dictionary.doc2bow(_tokenize_keep_all_words(left_text))
    vec = field_index.tfidf[bow]
    sims = field_index.index[vec]
    return np.asarray(sims, dtype=np.float32)


def _topk_from_sims(sims: np.ndarray, k: int) -> np.ndarray:
    n = sims.shape[0]
    if n == 0:
        return np.array([], dtype=np.int64)
    k = min(k, n)
    if k == n:
        return np.argsort(-sims)
    cand = np.argpartition(-sims, k)[:k]
    return cand[np.argsort(-sims[cand])]


def _year_equal(a, b) -> bool:
    if pd.isna(a) or pd.isna(b):
        return False
    try:
        return int(a) == int(b)
    except Exception:
        return str(a) == str(b)


# -----------------------------
# Thresholds + generator (Scenario 1 removed)
# -----------------------------
@dataclass
class Thresholds:
    s2_title_min: float = 0.80   # scenario 2: title + author
    s2_author_min: float = 0.80 # scenario 2: title + author
    s3_author_min: float = 0.85  # scenario 3: author + same year
    s4_title_min: float = 0.80   # scenario 4: title + same year


def generate_corner_case_pairs_gensim(
    left_df: pd.DataFrame,
    right_df: pd.DataFrame,
    *,
    scenario_counts: Dict[str, int],
    left_id_col: str = "id",
    right_id_col: str = "id",
    n_neighbors: int = 80,
    random_state: int = 42,
    smartirs: str = "ntc",
    weights: Tuple[float, float, float, float] = (0.45, 0.25, 0.15, 0.15),  # title, author, year, publisher
    thresholds: Thresholds = Thresholds(),
    max_attempts_multiplier: int = 30,
) -> pd.DataFrame:
    """
    Corner-case generator with gensim TF-IDF word unigrams, NO token deletions.

    Supported scenarios in scenario_counts:
      2_similar_titles_and_similar_author
      3_extremely_similar_author_same_year
      4_similar_title_same_year
      5_weighted_all_fields

    Output columns: id1/id2 + fields + similarity diagnostics.
    """
    rng = np.random.default_rng(random_state)

    L = _prep_two_df(left_df, "left")
    R = _prep_two_df(right_df, "right")

    if left_id_col not in L.columns:
        L[left_id_col] = np.arange(len(L))
    if right_id_col not in R.columns:
        R[right_id_col] = np.arange(len(R))

    # RIGHT indices per field (no deletions)
    idx_title = _build_gensim_tfidf_index_no_deletions(R["clean_title"].tolist(), smartirs=smartirs)
    idx_author = _build_gensim_tfidf_index_no_deletions(R["clean_author"].tolist(), smartirs=smartirs)
    idx_publisher = _build_gensim_tfidf_index_no_deletions(R["clean_publisher"].tolist(), smartirs=smartirs)

    # Caches
    title_sims_cache: Dict[int, np.ndarray] = {}
    author_sims_cache: Dict[int, np.ndarray] = {}
    publisher_sims_cache: Dict[int, np.ndarray] = {}

    def get_title_sims(i: int) -> np.ndarray:
        if i not in title_sims_cache:
            title_sims_cache[i] = _gensim_sims_for_left_text_no_deletions(idx_title, L.loc[i, "clean_title"])
        return title_sims_cache[i]

    def get_author_sims(i: int) -> np.ndarray:
        if i not in author_sims_cache:
            author_sims_cache[i] = _gensim_sims_for_left_text_no_deletions(idx_author, L.loc[i, "clean_author"])
        return author_sims_cache[i]

    def get_publisher_sims(i: int) -> np.ndarray:
        if i not in publisher_sims_cache:
            publisher_sims_cache[i] = _gensim_sims_for_left_text_no_deletions(idx_publisher, L.loc[i, "clean_publisher"])
        return publisher_sims_cache[i]

    def year_equal(i: int, j: int) -> bool:
        return _year_equal(L.loc[i, "publish_year"], R.loc[j, "publish_year"])

    def metrics(i: int, j: int) -> Dict[str, float]:
        st = float(get_title_sims(i)[j]) if idx_title.num_right > 0 else 0.0
        sa = float(get_author_sims(i)[j]) if idx_author.num_right > 0 else 0.0
        sp = float(get_publisher_sims(i)[j]) if idx_publisher.num_right > 0 else 0.0
        ye = 1.0 if year_equal(i, j) else 0.0
        w0, w1, w2, w3 = weights
        sw = w0 * st + w1 * sa + w2 * ye + w3 * sp
        return {
            "sim_title": st,
            "sim_author": sa,
            "sim_publisher": sp,
            "year_equal": float(ye),
            "sim_weighted": float(sw),
        }

    def quota(name: str) -> int:
        return int(scenario_counts.get(name, 0))

    counter = Counter()
    used = set()
    rows = []

    def try_add(i: int, j: int, scenario: str, m: Dict[str, float]) -> bool:
        if quota(scenario) <= 0 or counter[scenario] >= quota(scenario):
            return False
        key = (int(i), int(j), scenario)
        if key in used:
            return False

        rows.append({
            "scenario": scenario,
            "id1": L.loc[i, left_id_col],
            "id2": R.loc[j, right_id_col],
            "left_isbn_clean": L.loc[i, "isbn_clean"],
            "right_isbn_clean": R.loc[j, "isbn_clean"],
            "left_title": L.loc[i, "clean_title"],
            "right_title": R.loc[j, "clean_title"],
            "left_author": L.loc[i, "clean_author"],
            "right_author": R.loc[j, "clean_author"],
            "left_publisher": L.loc[i, "clean_publisher"],
            "right_publisher": R.loc[j, "clean_publisher"],
            "left_year": L.loc[i, "publish_year"],
            "right_year": R.loc[j, "publish_year"],
            **m
        })
        used.add(key)
        counter[scenario] += 1
        return True

    def done(s: str) -> bool:
        return counter[s] >= quota(s)

    # Scenario 2
    s2 = "2_similar_titles_and_similar_author"
    if quota(s2) > 0:
        attempts = 0
        max_attempts = quota(s2) * max_attempts_multiplier
        while not done(s2) and attempts < max_attempts:
            attempts += 1
            i = int(rng.integers(0, len(L)))
            t_sims = get_title_sims(i)
            a_sims = get_author_sims(i)
            topk = _topk_from_sims(t_sims, n_neighbors)
            for j in topk:
                if float(t_sims[j]) < thresholds.s2_title_min:
                    break
                if float(a_sims[j]) < thresholds.s2_author_min:
                    continue
                if try_add(i, int(j), s2, metrics(i, int(j))):
                    break

    # Scenario 3
    s3 = "3_extremely_similar_author_same_year"
    if quota(s3) > 0:
        attempts = 0
        max_attempts = quota(s3) * max_attempts_multiplier
        while not done(s3) and attempts < max_attempts:
            attempts += 1
            i = int(rng.integers(0, len(L)))
            a_sims = get_author_sims(i)
            topk = _topk_from_sims(a_sims, n_neighbors)
            for j in topk:
                if float(a_sims[j]) < thresholds.s3_author_min:
                    break
                if not year_equal(i, int(j)):
                    continue
                if try_add(i, int(j), s3, metrics(i, int(j))):
                    break

    # Scenario 4
    s4 = "4_similar_title_same_year"
    if quota(s4) > 0:
        attempts = 0
        max_attempts = quota(s4) * max_attempts_multiplier
        while not done(s4) and attempts < max_attempts:
            attempts += 1
            i = int(rng.integers(0, len(L)))
            t_sims = get_title_sims(i)
            topk = _topk_from_sims(t_sims, n_neighbors)
            for j in topk:
                if float(t_sims[j]) < thresholds.s4_title_min:
                    break
                if not year_equal(i, int(j)):
                    continue
                if try_add(i, int(j), s4, metrics(i, int(j))):
                    break

    # Scenario 5
    s5 = "5_weighted_all_fields"
    if quota(s5) > 0:
        buckets = [(0.85, 0.92), (0.70, 0.85), (0.55, 0.70)]
        target = quota(s5)
        per_bucket = max(1, target // len(buckets))

        attempts = 0
        max_attempts = target * max_attempts_multiplier * 2

        for lo, hi in buckets:
            filled = 0
            while filled < per_bucket and not done(s5) and attempts < max_attempts:
                attempts += 1
                i = int(rng.integers(0, len(L)))
                t_sims = get_title_sims(i)
                topk = _topk_from_sims(t_sims, n_neighbors)
                for j in topk:
                    m = metrics(i, int(j))
                    if lo <= m["sim_weighted"] < hi:
                        if try_add(i, int(j), s5, m):
                            filled += 1
                            break

        while not done(s5) and attempts < max_attempts:
            attempts += 1
            i = int(rng.integers(0, len(L)))
            t_sims = get_title_sims(i)
            topk = _topk_from_sims(t_sims, n_neighbors)
            for j in topk:
                m = metrics(i, int(j))
                if m["sim_weighted"] >= 0.55:
                    if try_add(i, int(j), s5, m):
                        break

    return pd.DataFrame(rows)

In [8]:
counts = {
    "2_similar_titles_and_similar_author": 100,
    "3_extremely_similar_author_same_year": 50,
    "4_similar_title_same_year": 50,
    "5_weighted_all_fields": 300,
}

corner_case_m2a = generate_corner_case_pairs_gensim(
    metabooks_sample, amazon_sample,
    scenario_counts=counts,
    left_id_col="id",
    right_id_col="id",
    n_neighbors=120,
)
corner_case_m2g = generate_corner_case_pairs_gensim(
    metabooks_sample, goodreads_sample,
    scenario_counts=counts,
    left_id_col="id",
    right_id_col="id",
    n_neighbors=120,

)
corner_case_g2a = generate_corner_case_pairs_gensim(
    goodreads_sample, amazon_sample,
    scenario_counts=counts,
    left_id_col="id",
    right_id_col="id",
    n_neighbors=120,
)

In [10]:
corner_case_g2a.sample(5)

,scenario,id1,id2,left_isbn_clean,right_isbn_clean,left_title,right_title,left_author,right_author,left_publisher,right_publisher,left_year,right_year,sim_title,sim_author,sim_publisher,year_equal,sim_weighted
484,5_weighted_all_fields,goodreads_9938,amazon_220317,015640057X,0156003902,henry and june from a journal of love the unex...,fire from a journal of love the unexpurgated d...,anas nin,anas nin,mariner books,harvest books,1990,1996,0.763369,1.0,0.051202,0.0,0.601197
301,5_weighted_all_fields,goodreads_7009,amazon_35087,0439682584,043965548X,harry potter boxed set books 15 harry potter 15,harry potter and the prisoner of azkaban harry...,j k rowling,j k rowling,scholastic,scholastic paperbacks,2004,2004,0.594496,1.0,0.687214,1.0,0.770605
155,4_similar_title_same_year,goodreads_50757,amazon_28891,0451411137,0451411137,men in kilts,men in kilts,katie macalister,katie macalister,onyx,onyx books,2003,2003,1.000000,1.0,0.973542,1.0,0.996031
189,4_similar_title_same_year,goodreads_17658,amazon_76212,0553379976,0553379976,gather together in my name,gather together in my name,maya angelou,maya angelou,bantam,bantam,1997,1997,1.000000,1.0,1.000000,1.0,1.000000
126,3_extremely_similar_author_same_year,goodreads_737,amazon_71420,0141439637,1592246605,the portrait of a lady,the turn of the screw,henry james,henry james,penguin classics,wildside press,2003,2003,0.037416,1.0,0.000000,1.0,0.416837


In [12]:
from __future__ import annotations

from typing import List
from collections import Counter
import re

import numpy as np
import pandas as pd

from gensim.corpora import Dictionary
from gensim.models import TfidfModel
from gensim.similarities import SparseMatrixSimilarity


# -----------------------------
# Prep
# -----------------------------
def _safe_str(x) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip()


def _prep_two_df(df: pd.DataFrame, side: str) -> pd.DataFrame:
    out = df.copy()
    out["__side"] = side
    for c in ["clean_title", "clean_author", "clean_publisher"]:
        if c not in out.columns:
            out[c] = ""
        out[c] = out[c].map(_safe_str)
    if "publish_year" not in out.columns:
        out["publish_year"] = np.nan
    if "isbn_clean" not in out.columns:
        out["isbn_clean"] = ""
    out["isbn_clean"] = out["isbn_clean"].map(_safe_str)
    return out


# -----------------------------
# Tokenisation (NO deletions)
# -----------------------------
_WORD_RE = re.compile(r"[0-9A-Za-z]+(?:'[0-9A-Za-z]+)?", re.UNICODE)


def _tokenize_keep_all_words(text: str) -> List[str]:
    if text is None:
        return []
    s = str(text).lower().strip()
    if not s:
        return []
    return _WORD_RE.findall(s)


# -----------------------------
# Gensim TF-IDF index (NO filter_extremes)
# -----------------------------
class GensimFieldIndex:
    def __init__(self, dictionary: Dictionary, tfidf: TfidfModel, index: SparseMatrixSimilarity, num_right: int):
        self.dictionary = dictionary
        self.tfidf = tfidf
        self.index = index
        self.num_right = num_right


def _build_gensim_tfidf_index_no_deletions(
    right_texts: List[str],
    *,
    smartirs: str = "ntc",
) -> GensimFieldIndex:
    right_tokens = [_tokenize_keep_all_words(t) for t in right_texts]
    dictionary = Dictionary(right_tokens)
    dictionary.compactify()

    num_right = len(right_tokens)

    if len(dictionary) == 0:
        dummy_tfidf = TfidfModel([[]], dictionary=dictionary, smartirs=smartirs)
        dummy_index = SparseMatrixSimilarity([[]], num_features=1)
        return GensimFieldIndex(dictionary, dummy_tfidf, dummy_index, num_right)

    right_bow = [dictionary.doc2bow(toks) for toks in right_tokens]
    tfidf = TfidfModel(right_bow, dictionary=dictionary, smartirs=smartirs)
    right_tfidf = tfidf[right_bow]
    index = SparseMatrixSimilarity(right_tfidf, num_features=len(dictionary))
    return GensimFieldIndex(dictionary, tfidf, index, num_right)


def _gensim_sims_for_left_text_no_deletions(field_index: GensimFieldIndex, left_text: str) -> np.ndarray:
    if field_index.num_right == 0:
        return np.zeros((0,), dtype=np.float32)
    if len(field_index.dictionary) == 0:
        return np.zeros((field_index.num_right,), dtype=np.float32)

    bow = field_index.dictionary.doc2bow(_tokenize_keep_all_words(left_text))
    vec = field_index.tfidf[bow]
    sims = field_index.index[vec]
    return np.asarray(sims, dtype=np.float32)


def _topk_from_sims(sims: np.ndarray, k: int) -> np.ndarray:
    n = sims.shape[0]
    if n == 0:
        return np.array([], dtype=np.int64)
    k = min(k, n)
    if k == n:
        return np.argsort(-sims)
    cand = np.argpartition(-sims, k)[:k]
    return cand[np.argsort(-sims[cand])]


def _year_equal(a, b) -> bool:
    if pd.isna(a) or pd.isna(b):
        return False
    try:
        return int(a) == int(b)
    except Exception:
        return str(a) == str(b)


def _year_diff_known(a, b) -> bool:
    if pd.isna(a) or pd.isna(b):
        return False
    return not _year_equal(a, b)
def generate_negative_pairs_gensim_two_datasets_isbn_simple(
    left_df: pd.DataFrame,
    right_df: pd.DataFrame,
    *,
    left_id_col: str = "id",
    right_id_col: str = "id",
    n_neighbors: int = 120,
    random_state: int = 42,
    smartirs: str = "ntc",
    # quotas
    n_title_high_author_low: int = 75,
    n_title_high_year_diff: int = 75,
    n_author_high_title_low: int = 75,          # NEW (replaces old C)
    n_random_nonmatches: int = 100,
    # thresholds
    title_high_min: float = 0.85,
    author_low_max: float = 0.35,
    author_high_min: float = 0.85,
    title_low_max_for_typeC: float = 0.35,      # NEW: "different title" gate
    max_attempts_multiplier: int = 60,
) -> pd.DataFrame:
    """
    Negative generator using gensim TF-IDF word unigrams, NO token deletions.

    Negative types:
      A) High title & low author
      B) High title & different publish year (both present)
      C) High author & different title  (NEW; replaces old C)
      D) Random non-matches (ISBN !=)

    ISBN rule:
      if left.isbn_clean == right.isbn_clean and both non-empty -> excluded
    """
    rng = np.random.default_rng(random_state)

    L = _prep_two_df(left_df, "left")
    R = _prep_two_df(right_df, "right")

    if left_id_col not in L.columns:
        L[left_id_col] = np.arange(len(L))
    if right_id_col not in R.columns:
        R[right_id_col] = np.arange(len(R))

    # RIGHT indices per field (no deletions)
    idx_title = _build_gensim_tfidf_index_no_deletions(R["clean_title"].tolist(), smartirs=smartirs)
    idx_author = _build_gensim_tfidf_index_no_deletions(R["clean_author"].tolist(), smartirs=smartirs)
    idx_publisher = _build_gensim_tfidf_index_no_deletions(R["clean_publisher"].tolist(), smartirs=smartirs)

    # Cache full similarity arrays per left row
    title_sims_cache = {}
    author_sims_cache = {}
    publisher_sims_cache = {}

    def get_title_sims(i: int) -> np.ndarray:
        if i not in title_sims_cache:
            title_sims_cache[i] = _gensim_sims_for_left_text_no_deletions(idx_title, L.loc[i, "clean_title"])
        return title_sims_cache[i]

    def get_author_sims(i: int) -> np.ndarray:
        if i not in author_sims_cache:
            author_sims_cache[i] = _gensim_sims_for_left_text_no_deletions(idx_author, L.loc[i, "clean_author"])
        return author_sims_cache[i]

    def get_publisher_sims(i: int) -> np.ndarray:
        if i not in publisher_sims_cache:
            publisher_sims_cache[i] = _gensim_sims_for_left_text_no_deletions(idx_publisher, L.loc[i, "clean_publisher"])
        return publisher_sims_cache[i]

    def is_isbn_match(i: int, j: int) -> bool:
        li = L.loc[i, "isbn_clean"]
        rj = R.loc[j, "isbn_clean"]
        return bool(li) and bool(rj) and (li == rj)

    def year_diff_known(i: int, j: int) -> bool:
        return _year_diff_known(L.loc[i, "publish_year"], R.loc[j, "publish_year"])

    used = set()  # (i, j, neg_type)
    rows = []
    counter = Counter()

    def add_pair(i: int, j: int, neg_type: str, st: float, sa: float, sp: float) -> bool:
        if is_isbn_match(i, j):
            return False
        key = (int(i), int(j), neg_type)
        if key in used:
            return False
        used.add(key)

        yi = L.loc[i, "publish_year"]
        yj = R.loc[j, "publish_year"]

        rows.append({
            "neg_type": neg_type,
            "id1": L.loc[i, left_id_col],
            "id2": R.loc[j, right_id_col],
            "left_isbn_clean": L.loc[i, "isbn_clean"],
            "right_isbn_clean": R.loc[j, "isbn_clean"],
            "left_title": L.loc[i, "clean_title"],
            "right_title": R.loc[j, "clean_title"],
            "left_author": L.loc[i, "clean_author"],
            "right_author": R.loc[j, "clean_author"],
            "left_publisher": L.loc[i, "clean_publisher"],
            "right_publisher": R.loc[j, "clean_publisher"],
            "left_year": yi,
            "right_year": yj,
            "sim_title": float(st),
            "sim_author": float(sa),
            "sim_publisher": float(sp),
            "year_equal": 1.0 if _year_equal(yi, yj) else 0.0,
            "year_diff_known": 1.0 if _year_diff_known(yi, yj) else 0.0,
        })
        counter[neg_type] += 1
        return True

    # A) High title, low author (retrieve by title)
    negA = "A_high_title_low_author"
    attempts = 0
    max_attempts = n_title_high_author_low * max_attempts_multiplier
    while counter[negA] < n_title_high_author_low and attempts < max_attempts:
        attempts += 1
        i = int(rng.integers(0, len(L)))
        t_sims = get_title_sims(i)
        a_sims = get_author_sims(i)
        topk = _topk_from_sims(t_sims, n_neighbors)

        for j in topk:
            j = int(j)
            if is_isbn_match(i, j):
                continue
            st = float(t_sims[j])
            if st < title_high_min:
                break
            sa = float(a_sims[j])
            if sa > author_low_max:
                continue
            sp = float(get_publisher_sims(i)[j])
            if add_pair(i, j, negA, st, sa, sp):
                break

    # B) High title, different year known (retrieve by title)
    negB = "B_high_title_year_diff"
    attempts = 0
    max_attempts = n_title_high_year_diff * max_attempts_multiplier
    while counter[negB] < n_title_high_year_diff and attempts < max_attempts:
        attempts += 1
        i = int(rng.integers(0, len(L)))
        t_sims = get_title_sims(i)
        topk = _topk_from_sims(t_sims, n_neighbors)

        for j in topk:
            j = int(j)
            if is_isbn_match(i, j):
                continue
            st = float(t_sims[j])
            if st < title_high_min:
                break
            if not year_diff_known(i, j):
                continue
            sa = float(get_author_sims(i)[j])
            sp = float(get_publisher_sims(i)[j])
            if add_pair(i, j, negB, st, sa, sp):
                break

    # C) High author, different title (retrieve by author)
    negC = "C_high_author_low_title"
    attempts = 0
    max_attempts = n_author_high_title_low * max_attempts_multiplier * 2
    while counter[negC] < n_author_high_title_low and attempts < max_attempts:
        attempts += 1
        i = int(rng.integers(0, len(L)))
        a_sims = get_author_sims(i)
        t_sims = get_title_sims(i)
        topk = _topk_from_sims(a_sims, n_neighbors)

        for j in topk:
            j = int(j)
            if is_isbn_match(i, j):
                continue

            sa = float(a_sims[j])
            if sa < author_high_min:
                break  # sorted desc

            st = float(t_sims[j])
            if st > title_low_max_for_typeC:
                continue  # not "different title" enough

            sp = float(get_publisher_sims(i)[j])
            if add_pair(i, j, negC, st, sa, sp):
                break

    # D) Random non-matches
    negD = "D_random_nonmatches"
    attempts = 0
    max_attempts = n_random_nonmatches * max_attempts_multiplier * 10
    while counter[negD] < n_random_nonmatches and attempts < max_attempts:
        attempts += 1
        i = int(rng.integers(0, len(L)))
        j = int(rng.integers(0, len(R)))
        if is_isbn_match(i, j):
            continue
        st = float(get_title_sims(i)[j]) if idx_title.num_right else 0.0
        sa = float(get_author_sims(i)[j]) if idx_author.num_right else 0.0
        sp = float(get_publisher_sims(i)[j]) if idx_publisher.num_right else 0.0
        add_pair(i, j, negD, st, sa, sp)

    return pd.DataFrame(rows)

In [13]:
neg_m2a = generate_negative_pairs_gensim_two_datasets_isbn_simple(
    metabooks_sample,
    amazon_sample)
neg_m2g = generate_negative_pairs_gensim_two_datasets_isbn_simple(
    metabooks_sample,
    goodreads_sample)
neg_g2a = generate_negative_pairs_gensim_two_datasets_isbn_simple(
    goodreads_sample,
    amazon_sample)

In [14]:
neg_g2a.sample(5)

,neg_type,id1,id2,left_isbn_clean,right_isbn_clean,left_title,right_title,left_author,right_author,left_publisher,right_publisher,left_year,right_year,sim_title,sim_author,sim_publisher,year_equal,year_diff_known
165,C_high_author_low_title,goodreads_10928,amazon_59144,014101587X,0679455639,the confessor,the mark of the assassin,daniel silva,daniel silva,penguin,random house inc,2004,1998,0.015411,1.0,0.0,0.0,1.0
35,A_high_title_low_author,goodreads_20190,amazon_224373,174331731X,044104770X,barracuda,barracuda,christos tsiolkas,greenfield,allen unwin,berkley pub group mm,2013,1979,1.000000,0.0,0.0,0.0,1.0
51,A_high_title_low_author,goodreads_39271,amazon_66623,1909192791,0064400735,its like this,its like this cat,anne ogleadra,emily cheney neville,beaten track publishing,harpertrophy,2014,1975,0.873325,0.0,0.0,0.0,1.0
63,A_high_title_low_author,goodreads_12959,amazon_32179,0515143189,0312973063,safe harbor,safe harbor,christine feehan,antoinette stockenberg,jove,st martins press,2007,2000,1.000000,0.0,0.0,0.0,1.0
8,A_high_title_low_author,goodreads_1515,amazon_38692,0385733410,0140062718,rebel angels,the rebel angels,libba bray,robertson davies,ember,penguin books,2006,1983,0.995540,0.0,0.0,0.0,1.0


In [15]:
def pair_exact_matches(
    df_left: pd.DataFrame,
    left_id_col: str,
    df_right: pd.DataFrame,
    right_id_col: str,
    *,
    n_matches: int,
    random_state: int = 42,
    one_to_one: bool = True,
) -> pd.DataFrame:
    """
    Exact matches via isbn_clean equality with a fixed number of returned pairs.

    Parameters
    ----------
    n_matches:
        Exact number of matches to return (if available).
    one_to_one:
        If True, enforce 1–1 matching (no left or right ID reused).
        If False, allow many-to-many within same ISBN.

    Returns
    -------
    DataFrame with columns:
      id_left, id_right, label=1
    """
    rng = np.random.default_rng(random_state)

    L = (
        df_left
        .dropna(subset=["isbn_clean"])
        [[left_id_col, "isbn_clean"]]
        .rename(columns={left_id_col: "id1", "isbn_clean": "isbn"})
    )
    R = (
        df_right
        .dropna(subset=["isbn_clean"])
        [[right_id_col, "isbn_clean"]]
        .rename(columns={right_id_col: "id2", "isbn_clean": "isbn"})
    )

    # All possible exact matches
    merged = (
        L.merge(R, on="isbn", how="inner")
         .loc[:, ["id1", "id2", "isbn"]]
         .drop_duplicates()
    )

    if merged.empty:
        return pd.DataFrame(columns=["id1", "id2", "label"])

    # Shuffle candidates
    merged = merged.sample(frac=1.0, random_state=random_state).reset_index(drop=True)

    pairs = []
    used_left = set()
    used_right = set()

    for row in merged.itertuples(index=False):
        if one_to_one:
            if row.id1 in used_left or row.id2 in used_right:
                continue

        pairs.append((row.id1, row.id2))
        used_left.add(row.id1)
        used_right.add(row.id2)

        if len(pairs) >= n_matches:
            break

    out = pd.DataFrame(pairs, columns=["id1", "id2"])
    out["label"] = 1
    return out

In [16]:
pos_m2a = pair_exact_matches(
    df_left=metabooks_sample,
    df_right=amazon_sample,
    n_matches=400,
    left_id_col="id",
    right_id_col="id",
    one_to_one=True,
)
pos_m2g = pair_exact_matches(
    df_left=metabooks_sample,
    df_right=goodreads_sample,
    n_matches=400,
    left_id_col="id",
    right_id_col="id",
    one_to_one=True,
)
pos_g2a = pair_exact_matches(
    df_left=goodreads_sample,
    df_right=amazon_sample,
    n_matches=400,
    left_id_col="id",
    right_id_col="id",
    one_to_one=True,
)

In [17]:
# For corner case df we check left_isbn_clean and right_isbn_clean to set label
corner_case_m2a['label'] = (corner_case_m2a["right_isbn_clean"]== corner_case_m2a["left_isbn_clean"]).astype(int)
corner_case_m2g['label'] = (corner_case_m2g["right_isbn_clean"]== corner_case_m2g["left_isbn_clean"]).astype(int)
corner_case_g2a['label'] = (corner_case_g2a["right_isbn_clean"]== corner_case_g2a["left_isbn_clean"]).astype(int)
neg_m2a['label'] = (neg_m2a["right_isbn_clean"]== neg_m2a["left_isbn_clean"]).astype(int)
neg_m2g['label'] = (neg_m2g["right_isbn_clean"]== neg_m2g["left_isbn_clean"]).astype(int)
neg_g2a['label'] = (neg_g2a["right_isbn_clean"]== neg_g2a["left_isbn_clean"]).astype(int)

In [18]:
corner_case_g2a.groupby('scenario')['label'].value_counts()

scenario                              label
2_similar_titles_and_similar_author   0         64
                                      1         36
3_extremely_similar_author_same_year  0         28
                                      1         22
4_similar_title_same_year             1         32
                                      0         14
5_weighted_all_fields                 0        185
                                      1        115
Name: count, dtype: int64

In [19]:
neg_m2a.drop_duplicates(subset=['id1', 'id2'], inplace=True, keep='first')
neg_m2g.drop_duplicates(subset=['id1', 'id2'], inplace=True, keep='first')
neg_g2a.drop_duplicates(subset=['id1', 'id2'], inplace=True, keep='first')
corner_case_m2a.drop_duplicates(subset=['id1', 'id2'], inplace=True, keep='first')
corner_case_m2g.drop_duplicates(subset=['id1', 'id2'], inplace=True, keep='first')
corner_case_g2a.drop_duplicates(subset=['id1', 'id2'], inplace=True, keep='first')

In [20]:
def finalize_and_split(pos_df, corners_labeled_df, neg_df, *, test_frac=0.2, seed=42):
    """
    Merge positives + labeled corners + negatives, then stratified split (train/test only).
    Returns:
        train_df, test_df  (each with id_left, id_right, label)
    """
    # Merge all labeled pairs
    all_labeled = pd.concat(
        [
            pos_df[["id1", "id2", "label"]],
            corners_labeled_df[["id1", "id2", "label"]],
            neg_df[["id1", "id2", "label"]],
        ],
        ignore_index=True
    ).dropna(subset=["label"]).drop_duplicates()

    # Ensure binary integer labels
    all_labeled["label"] = all_labeled["label"].astype(int)

    # Shuffle once for randomness
    L = all_labeled.sample(frac=1.0, random_state=seed).reset_index(drop=True)

    # Stratified split (preserve 0/1 balance)
    parts = {"train": [], "test": []}
    for y in (0, 1):
        grp = L[L["label"] == y]
        n = len(grp)
        n_test = int(round(n * test_frac))
        test_part = grp.iloc[:n_test]
        train_part = grp.iloc[n_test:]
        parts["train"].append(train_part)
        parts["test"].append(test_part)

    train = pd.concat(parts["train"], ignore_index=True)
    test  = pd.concat(parts["test"],  ignore_index=True)
    return train, test

In [21]:
train_MA, test_MA =finalize_and_split(pos_m2a,neg_m2a[["id1","id2","label"]],corner_case_m2a[["id1","id2","label"]],test_frac=0.2,seed=42)
train_MG,test_MG =finalize_and_split(pos_m2g,neg_m2g[["id1","id2","label"]],corner_case_m2g[["id1","id2","label"]],test_frac=0.2,seed=42)
train_GA,test_GA =finalize_and_split(pos_g2a,neg_g2a[["id1","id2","label"]],corner_case_g2a[["id1","id2","label"]],test_frac=0.2,seed=42)

In [22]:
display(train_MG.label.describe())
display(test_MG.label.describe())
display(train_GA.label.describe())
# Stratified split

count    956.000000
mean       0.601464
std        0.489853
min        0.000000
25%        0.000000
50%        1.000000
75%        1.000000
max        1.000000
Name: label, dtype: float64

count    239.000000
mean       0.602510
std        0.490406
min        0.000000
25%        0.000000
50%        1.000000
75%        1.000000
max        1.000000
Name: label, dtype: float64

count    926.000000
mean       0.481641
std        0.499933
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        1.000000
Name: label, dtype: float64

In [24]:
train_GA.to_csv(ROOT/'ml-datasets/train_GA.csv', index=False)
test_GA.to_csv(ROOT/'ml-datasets/test_GA.csv', index=False)

In [148]:
train_MG.to_csv(ROOT/"ml-datasets/train_MG.csv",index=False)
test_MG.to_csv(ROOT/"ml-datasets/test_MG.csv",index=False)
train_MA.to_csv(ROOT/"ml-datasets/train_MA.csv",index=False)
test_MA.to_csv(ROOT/"ml-datasets/test_MA.csv",index=False)